In [1]:
import pandas as pd
import math
from collections import Counter

In [2]:
data = pd.read_csv("../../dataset/news_preprocess.csv")
data

,news_id,Judul,Content
0,1,jokowi kena pakai adat betawi sidang tahun akh...,jakarta kompascom presiden joko widodo pakai b...
1,2,amnesty international kan indikator krisis dem...,tempoco jakarta amnesty international indonesi...
2,3,jelang long weekend stasiun kereta cepat halim...,stasiun kereta cepat whoosh halim jakarta timu...
3,4,kpu tegas pilih tak daftar dpt nyoblos begini ...,jakarta kompascom komisi pilih umum kpu pasti ...
4,5,kemenag luncur gera senam haji jaga tahan fisi...,tempoco jakarta menteri agama kemenag luncur s...
...,...,...,...
95,96,program prioritas kemendikbud guru guru gerak ...,tempoco riau bijak merdeka ajar episode jumlah...
96,97,cerita korban tipu visa haji ilegal panas ding...,mek kompascom lukman bukan nama benar orang gu...
97,98,gerindra sebut reshuffle menkumham sinkronisas...,jakarta kompascom ketua hari partai gerindra s...
98,99,lawrence wong lantik jadi singapura,lawrence wong lantik perdana menteri singapura...


In [3]:
class CbfService:

    def __init__(self, input_csv="../../dataset/news_preprocess.csv"):
        self.input_csv = input_csv

    def compute_tf(self, text):

        words = text.split()

        freq = Counter(words)

        if len(freq) > 0:
            max_freq = max(freq.values())
        else:
            max_freq = 1

        tf = {}

        for word in freq:
            tf[word] = freq[word] / max_freq

        # =========================
        # SAVE CSV TF
        # =========================
        tf_df = pd.DataFrame([
            {
                "word": word,
                "tf": value
            }
            for word, value in tf.items()
        ])

        tf_df.to_csv(
            "../../dataset/experiment/tf_output.csv",
            index=False
        )

        return tf

    def compute_idf(self, documents):

        N = len(documents)

        word_doc_count = {}

        for text in documents:

            words = text.split()

            unique_words = set(words)

            for word in unique_words:

                if word in word_doc_count:
                    word_doc_count[word] += 1
                else:
                    word_doc_count[word] = 1

        idf = {}

        for word in word_doc_count:

            ni = word_doc_count[word]

            idf[word] = math.log(N / ni)

        # =========================
        # SAVE CSV IDF
        # =========================
        idf_df = pd.DataFrame([
            {
                "word": word,
                "idf": value
            }
            for word, value in idf.items()
        ])

        idf_df.to_csv(
            "../../dataset/experiment/idf_output.csv",
            index=False
        )

        return idf

    def compute_tfidf_vector(self, tf_dict, idf_dict):

        tfidf = {}

        for word in tf_dict:

            if word in idf_dict:
                tfidf[word] = tf_dict[word] * idf_dict[word]
            else:
                tfidf[word] = 0

        # =========================
        # SAVE CSV TF-IDF VECTOR
        # =========================
        tfidf_df = pd.DataFrame([
            {
                "word": word,
                "tfidf": value
            }
            for word, value in tfidf.items()
        ])

        tfidf_df.to_csv(
            "../../dataset/experiment/tfidf_vector_output.csv",
            index=False
        )

        return tfidf

    def cosine_similarity(self, vec1, vec2):

        dot_product = 0

        for word in vec1:

            if word in vec2:
                dot_product += vec1[word] * vec2[word]

        sum1 = 0

        for v in vec1.values():
            sum1 += v * v

        norm1 = math.sqrt(sum1)

        sum2 = 0

        for v in vec2.values():
            sum2 += v * v

        norm2 = math.sqrt(sum2)

        if norm1 == 0:
            return 0

        if norm2 == 0:
            return 0

        similarity = dot_product / (norm1 * norm2)

        # =========================
        # SAVE CSV COSINE SIMILARITY
        # =========================
        cosine_df = pd.DataFrame([
            {
                "cosine_similarity": similarity
            }
        ])

        cosine_df.to_csv(
            "../../dataset/experiment/cosine_similarity_output.csv",
            index=False
        )

        return similarity

    def compute_tfidf(self):

        df = pd.read_csv(self.input_csv)

        df = df.reset_index(drop=True)

        df["cbf_text"] = (
            df["Judul"].fillna("") + " " +
            df["Content"].fillna("")
        )

        tf_list = []

        for text in df["cbf_text"]:

            tf = self.compute_tf(text)

            tf_list.append(tf)

        df["TF"] = tf_list

        idf = self.compute_idf(df["cbf_text"])

        tfidf_list = []

        for tf in df["TF"]:

            tfidf = self.compute_tfidf_vector(tf, idf)

            tfidf_list.append(tfidf)

        df["TF_IDF"] = tfidf_list

        output_path = "../../dataset/experiment/news_tfidf.csv"

        df.to_csv(output_path, index=False)

        return df

    def recommendation(self, news_id, candidate_size=20):

        df = self.compute_tfidf()

        target_index = None

        for i in range(len(df)):

            if df["news_id"].iloc[i] == news_id:
                target_index = i
                break

        if target_index is None:
            raise ValueError("News ID tidak ditemukan")

        target_vec = df["TF_IDF"].iloc[target_index]

        results = []

        for i in range(len(df)):

            if i == target_index:
                continue

            vec = df["TF_IDF"].iloc[i]

            sim = self.cosine_similarity(target_vec, vec)

            result = {
                "news_id": news_id,
                "similar_news_id": df["news_id"].iloc[i],
                "title": df["Judul"].iloc[i],
                "score": float(sim)
            }

            results.append(result)

        results = sorted(
            results,
            key=lambda item: item["score"],
            reverse=True
        )

        # =========================
        # SAVE CSV RECOMMENDATION
        # =========================
        recommendation_df = pd.DataFrame(results)

        recommendation_df.to_csv(
            "../../dataset/experiment/recommendation_output.csv",
            index=False
        )

        return results[:candidate_size]

In [4]:
service = CbfService()

In [5]:
df = service.compute_tfidf()

In [6]:
data_tf = pd.read_csv("../../dataset/experiment/tf_output.csv")
data_tf

,word,tf
0,kpu,0.8125
1,jakarta,1.0000
2,gelar,0.1875
3,rapat,0.6250
4,pleno,0.5625
...,...,...
150,perundangundangan,0.0625
151,pasti,0.0625
152,prosedur,0.0625
153,jalan,0.0625


In [7]:
data_idf = pd.read_csv("../../dataset/experiment/fix_idf_output.csv")
data_idf

,word,idf
0,kpu,1.771957
1,jakarta,0.301105
2,gelar,1.771957
3,rapat,1.714798
4,pleno,2.995732
5,agustus,1.966113
6,bahas,1.966113
7,status,3.912023
8,dharma,3.912023
9,pongrekun,4.605170


In [8]:
data_tf_idf = pd.read_csv("../../dataset/experiment/tfidf_vector_output.csv")
data_tf_idf

,word,tfidf
0,kpu,1.439715
1,jakarta,0.301105
2,gelar,0.332242
3,rapat,1.071749
4,pleno,1.685099
...,...,...
150,perundangundangan,0.287823
151,pasti,0.089195
152,prosedur,0.219160
153,jalan,0.062141


In [9]:
service = CbfService()

df = service.compute_tfidf()

rows = []

for i in range(len(df)):
    for j in range(len(df)):

        if i == j:
            continue

        vec1 = df["TF_IDF"].iloc[i]
        vec2 = df["TF_IDF"].iloc[j]

        sim = service.cosine_similarity(vec1, vec2)

        rows.append({
            "news_id": df["news_id"].iloc[i],
            "similar_news_id": df["news_id"].iloc[j],
            "score": sim
        })

similarity_df = pd.DataFrame(rows)

similarity_df.to_csv(
    "../../dataset/experiment/all_similarity_output.csv",
    index=False
)

similarity_df

,news_id,similar_news_id,score
0,1,2,0.022751
1,1,3,0.003745
2,1,4,0.013939
3,1,5,0.010383
4,1,6,0.010593
...,...,...,...
9895,100,95,0.015278
9896,100,96,0.013406
9897,100,97,0.004191
9898,100,98,0.018260
